# Task 2 - Total number of balls on top of the table

## Imports

In [ ]:
# ─── Standard Library ───────────────────────────────────────
import os
import json
import time
import random
import warnings
from dotenv import load_dotenv
load_dotenv()
warnings.filterwarnings("ignore")

# ─── Numeric / Data ─────────────────────────────────────────
import numpy as np
import pandas as pd

# ─── Image Processing ───────────────────────────────────────
import cv2
from PIL import Image

# ─── Visualization ──────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns

# ─── PyTorch Core ───────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# ─── Torchvision ────────────────────────────────────────────
import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.v2 as T          # newer augmentation API
from torchvision import models

# ─── Metrics ────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

# ─── Roboflow (dataset download) ────────────────────────────
from roboflow import Roboflow

# ─── Misc Utilities ─────────────────────────────────────────
from tqdm.notebook import tqdm         # progress bars in Jupyter
from pathlib import Path
import itertools
import copy

# ─── Device Setup ───────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cpu


In [ ]:
# ─── DATASET DOWNLOAD ───────────────────────────────────────
DOWNLOAD_DATASETS = False

if DOWNLOAD_DATASETS:
    rf = Roboflow(api_key=os.getenv("ROBOFLOW_API_KEY"))

    datasets_dir = Path("datasets")
    datasets_dir.mkdir(exist_ok=True)

    # Original working dir
    original_dir = os.getcwd()
    
    all_datasets = [
        ("bachelorthesis",            "8-ball-pool-l530o"),        # main
        ("nidacorian-protonmail-com", "pool-billiard"),             # extra 1
        ("mark-dj0yk",               "pool-balls-detection-srlqi"),# extra 2
        ("pool-ball-detection",      "pool-ball-detection-6lfd9"), # extra 3
    ]

    for workspace_name, project_name in all_datasets:
        print(f"\n{'='*50}")
        print(f"Downloading: {workspace_name}/{project_name}")
        print(f"{'='*50}")
        
        try:
            project = rf.workspace(workspace_name).project(project_name)
            versions = project.versions()
            print(f"Found {len(versions)} version(s)")
            
            # Each dataset goes into its own subfolder
            project_dir = datasets_dir / project_name
            project_dir.mkdir(exist_ok=True)
            os.chdir(project_dir)
            
            for v in versions:
                print(f"  -> Downloading version {v.version}...")
                v.download("coco")
            
        except Exception as e:
            print(f"  [ERROR] Failed to download {project_name}: {e}")
        
        finally:
            os.chdir(original_dir)

    print(f"\nAll done! Datasets saved in: {datasets_dir.resolve()}")


Downloading: bachelorthesis/8-ball-pool-l530o
loading Roboflow workspace...
loading Roboflow project...
Found 2 version(s)
  -> Downloading version 3...



Extracting Dataset Version Zip to 8-Ball-Pool-3 in coco:: 100%|██████████| 251/251 [00:00<00:00, 2802.78it/s]


  -> Downloading version 2...



Extracting Dataset Version Zip to 8-Ball-Pool-2 in coco:: 100%|██████████| 203/203 [00:00<00:00, 3700.87it/s]


Downloading: nidacorian-protonmail-com/pool-billiard
loading Roboflow workspace...


...

  -> Downloading version 3...
Exporting format coco in progress : 30.0%
Version export complete for coco format



Extracting Dataset Version Zip to Pool-Ball-Detection-3 in coco:: 100%|██████████| 783/783 [00:00<00:00, 6811.10it/s]


  -> Downloading version 2...
Exporting format coco in progress : 95.0%
Version export complete for coco format



Extracting Dataset Version Zip to Pool-Ball-Detection-2 in coco:: 100%|██████████| 1095/1095 [00:00<00:00, 9694.94it/s] 


All done! Datasets saved in: /home/admin/Documents/FEUP-CV/proj/datasets
